# Graph-Based Product Recommendation Engine
## Comparing Graph-Based and Traditional Recommendation Methods
### Scalable Data Science Mini Project · Phase 1 (Backend, Google Colab)

**Dataset:** Retailrocket e-commerce dataset (`events.csv`)

**Methods compared**
1. **Item-Item Collaborative Filtering** (traditional baseline): cosine similarity on a user-item matrix
2. **Common Neighbors** (graph-based): similar users through shared products
3. **Personalized PageRank** (graph-based): random walk with restart on the user-product graph

**How to run**
1. `Runtime → Change runtime type → CPU` (no GPU needed).
2. In **Section 2**, set `DATA_SOURCE` to `"upload"` or `"drive"`.
3. `Runtime → Run all`. The only input needed is the file upload or Drive permission.
4. Download `outputs.zip` at the end (Section 11). It is what the Streamlit dashboard will use in Phase 2.

**Expected runtime:** roughly 10–30 minutes on a Colab CPU, depending on graph size. Personalized PageRank is the slow step. To make it faster, lower `N_EVAL_USERS` / `N_SCALE_EVAL_USERS` in the config cell.

**Contents**
1. Environment setup
2. Load the dataset
3. Dataset inspection
4. Preprocessing: weights, cleaning, chronological split, filtering
5. Baseline: Item-Item Collaborative Filtering
6. Graph construction (NetworkX bipartite graph)
7. Graph-based recommenders: Common Neighbors and Personalized PageRank
8. Evaluation and comparison
9. Scalability analysis
10. Backend validation (tests)
11. Export outputs for Phase 2
12. Viva notes

> Every number in this notebook is computed from the data while it runs. Nothing is hardcoded.

---
# 1. Environment Setup
Colab already includes most of these libraries. The cell below installs anything missing (quietly).

In [ ]:
# Install / upgrade the required libraries (quiet). Safe to re-run.
!pip install -q networkx psutil plotly seaborn scikit-learn

In [ ]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
import os, sys, gc, json, time, math, glob, heapq, random, pickle, shutil, zipfile, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
import psutil
import sklearn
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid")

# Are we inside Google Colab?
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Python      : {sys.version.split()[0]}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"networkx    : {nx.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"plotly      : {plotly.__version__}")
print(f"Running in Colab: {IN_COLAB}")
print(f"Available RAM   : {psutil.virtual_memory().total / 1e9:.1f} GB")

### Project configuration
All tunable settings are in one place. The same values are saved to `config.json` so the dashboard can show exactly how the results were produced.

In [ ]:
# ---------------------------------------------------------------
# Global configuration
# ---------------------------------------------------------------
SEED = 42                      # reproducibility for every random choice
np.random.seed(SEED)
random.seed(SEED)

# Interaction weights (explained in Section 4)
EVENT_WEIGHTS = {"view": 1, "addtocart": 3, "transaction": 5}
MAX_EDGE_WEIGHT = 15           # cap on the summed weight of one user-item pair (limits repeat-view spam)

# Chronological split
TRAIN_FRACTION = 0.80          # earliest 80% of events (by time) = train, latest 20% = test

# Filtering (computed on TRAIN only, to avoid leakage)
MIN_USER_ITEMS = 5             # a user needs >= 5 distinct products in train
MIN_ITEM_USERS = 5             # a product needs >= 5 distinct users in train
MAX_USER_ITEMS = 500           # users with more distinct products than this are treated as bots/crawlers

# Recommenders
TOP_K = 5                      # recommendations per user
MIN_HISTORY = 2                # below this many products we fall back to popularity
CF_NEIGHBORS = 50              # keep the 50 most similar items per item (k-NN item similarity)
CN_TOP_USERS = 50              # Common Neighbors: use the 50 most similar users
PPR_ALPHA = 0.85               # PageRank damping factor (restart probability = 1 - alpha = 0.15)
PPR_MAX_ITER = 100
PPR_TOL = 1e-6

# Evaluation
N_EVAL_USERS = 300             # users sampled (reproducibly) for evaluation; PPR is the slow part
N_BOOTSTRAP = 1000             # bootstrap resamples for 95% confidence intervals

# Scalability
SCALE_FRACTIONS = [0.10, 0.30, 0.60, 1.00]
N_SCALE_EVAL_USERS = 200       # max users evaluated at each dataset size (fewer if a sample has fewer)

# Output folders (same structure as the Phase 2 project)
OUTPUT_DIR = "/content/outputs" if IN_COLAB else "outputs"
DATA_OUT   = os.path.join(OUTPUT_DIR, "data")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
VIS_DIR    = os.path.join(OUTPUT_DIR, "visualizations")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
for d in (DATA_OUT, RESULTS_DIR, VIS_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)

METHOD_COLORS = {
    "Item-Item CF": "#4C72B0",
    "Common Neighbors": "#DD8452",
    "Personalized PageRank": "#55A868",
    "Popularity (reference)": "#8C8C8C",
}

def save_fig(name):
    """Save the current matplotlib figure into visualizations/ and display it."""
    path = os.path.join(VIS_DIR, name)
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved -> {path}")

def save_plotly(fig, name):
    """Save a Plotly figure as interactive HTML into visualizations/ and display it."""
    path = os.path.join(VIS_DIR, name)
    fig.write_html(path, include_plotlyjs="cdn")
    fig.show()
    print(f"Saved -> {path}")

def rss_mb():
    """Current memory used by this Python process (MB)."""
    return psutil.Process(os.getpid()).memory_info().rss / 1e6

print("Configuration loaded. Outputs will be written to:", OUTPUT_DIR)

---
# 2. Load the Dataset
The dataset is on your Windows PC, and Colab cannot read Windows paths such as `C:\Users\...`. Choose one option:

| `DATA_SOURCE` | What happens | When to use |
|---|---|---|
| `"upload"` | A file picker opens; choose the Kaggle zip (`archive.zip`) or `events.csv` | Simplest. The ~300 MB zip can take a while; keep the tab in the foreground |
| `"drive"` | Mounts Google Drive and reads `DRIVE_PATH` (zip or CSV) | Most reliable. Upload to Drive once, reuse on every run |
| `"local"` | Reads from a local folder | Only when running outside Colab (e.g. VS Code/Jupyter) |

Only `events.csv` is required. The other Retailrocket files (`item_properties_part1/2.csv`, `category_tree.csv`) describe product attributes and are **not needed** for interaction-based recommenders. The cell reports whether they are present.

In [ ]:
# ---------------------------------------------------------------
# CHOOSE YOUR DATA SOURCE HERE (the Kaggle zip can be used as-is)
# ---------------------------------------------------------------
# "upload" -> file picker: choose the Kaggle zip (e.g. archive.zip) or events.csv
# "drive"  -> read the zip/csv from Google Drive (more reliable for the ~300 MB zip)
# "local"  -> a path on this machine (only when running outside Colab)
DATA_SOURCE = "upload"
DRIVE_PATH = "/content/drive/MyDrive/retailrocket/archive.zip"   # zip or events.csv in Drive
LOCAL_PATH = "data/events.csv"                                   # zip or events.csv, for "local"

WORK_DATA_DIR = "/content/data" if IN_COLAB else "data_work"
os.makedirs(WORK_DATA_DIR, exist_ok=True)
OPTIONAL_FILES = ["item_properties_part1.csv", "item_properties_part2.csv", "category_tree.csv"]


def extract_events_from_zip(zip_path, dest_dir):
    # Extract only events.csv (works even if it sits inside a subfolder of the zip)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"{zip_path} is not a valid zip file. Re-download it from Kaggle.")
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.namelist()
        print("Files inside the zip:")
        for m in members:
            print(f"  {m:<45} {zf.getinfo(m).file_size/1e6:8.1f} MB")
        target = None
        for m in members:
            if os.path.basename(m).lower() == "events.csv":
                target = os.path.join(dest_dir, "events.csv")
                with zf.open(m) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)          # streamed to disk, low memory use
        for opt in OPTIONAL_FILES:
            present = any(os.path.basename(m).lower() == opt for m in members)
            print(f"  optional {opt:<28}: {'in zip (not needed, skipped)' if present else 'not in zip'}")
    if target is None:
        raise FileNotFoundError("events.csv not found inside the zip. Is this the Retailrocket dataset zip?")
    return target


if DATA_SOURCE == "upload":
    if not IN_COLAB:
        raise RuntimeError('DATA_SOURCE="upload" only works in Google Colab. Use "local".')
    from google.colab import files
    print("Choose the Retailrocket zip from Kaggle (e.g. archive.zip) or events.csv...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Re-run this cell.")
    fname = next(iter(uploaded))
    src_path = os.path.join(WORK_DATA_DIR, fname)
    if os.path.exists(fname):
        shutil.move(fname, src_path)                      # Colab saves uploads to the current folder
    else:
        with open(src_path, "wb") as fh:
            fh.write(uploaded[fname])
    del uploaded; gc.collect()
    print(f"Uploaded {fname} ({os.path.getsize(src_path)/1e6:.1f} MB)")
elif DATA_SOURCE == "drive":
    if not IN_COLAB:
        raise RuntimeError('DATA_SOURCE="drive" only works in Google Colab. Use "local".')
    from google.colab import drive
    drive.mount("/content/drive")
    src_path = DRIVE_PATH
elif DATA_SOURCE == "local":
    src_path = LOCAL_PATH
else:
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}")

if not os.path.exists(src_path):
    raise FileNotFoundError(f"File not found: {src_path}")
EVENTS_PATH = src_path if src_path.lower().endswith(".csv") else extract_events_from_zip(src_path, WORK_DATA_DIR)
print(f"\nevents.csv ready at: {EVENTS_PATH} ({os.path.getsize(EVENTS_PATH)/1e6:.1f} MB)")

In [ ]:
# ---------------------------------------------------------------
# Load events.csv and verify its structure
# ---------------------------------------------------------------
REQUIRED_COLUMNS = ["timestamp", "visitorid", "event", "itemid"]

try:
    events_raw = pd.read_csv(EVENTS_PATH)
except Exception as e:
    raise RuntimeError(f"Could not read {EVENTS_PATH}: {e}")

# Normalise column names (guards against stray spaces/capitals)
events_raw.columns = [c.strip().lower() for c in events_raw.columns]
missing_cols = [c for c in REQUIRED_COLUMNS if c not in events_raw.columns]
if missing_cols:
    raise ValueError(f"events.csv is missing required columns {missing_cols}. Found: {list(events_raw.columns)}")

print("Shape          :", events_raw.shape)
print("Columns        :", list(events_raw.columns))
print("Memory usage   : %.1f MB" % (events_raw.memory_usage(deep=True).sum() / 1e6))
print("\nData types:")
print(events_raw.dtypes)
display(events_raw.head(10))

---
# 3. Dataset Inspection
Before modelling we check data quality (missing values, duplicates), how the events are distributed, and how active users are.

In [ ]:
# ---------------------------------------------------------------
# Missing values & duplicates
# ---------------------------------------------------------------
missing = pd.DataFrame({
    "missing_count": events_raw.isna().sum(),
    "missing_pct": (events_raw.isna().mean() * 100).round(2),
})
print("Missing values per column:")
display(missing)
if "transactionid" in events_raw.columns:
    print("Note: transactionid is only filled for 'transaction' events, so missing values there are expected.\n")

n_exact_dupes = events_raw.duplicated().sum()
n_key_dupes = events_raw.duplicated(subset=REQUIRED_COLUMNS).sum()
print(f"Exact duplicate rows                                : {n_exact_dupes:,}")
print(f"Duplicate (timestamp, visitorid, event, itemid) rows: {n_key_dupes:,}")

In [ ]:
# ---------------------------------------------------------------
# Interaction types, users, products, time range
# ---------------------------------------------------------------
event_counts = events_raw["event"].value_counts()
event_dist = pd.DataFrame({"count": event_counts, "pct": (event_counts / len(events_raw) * 100).round(3)})
print("Interaction type distribution:")
display(event_dist)

ts = events_raw["timestamp"]
ts_unit = "ms" if ts.max() > 1e12 else "s"           # Retailrocket uses Unix milliseconds
t_min, t_max = pd.to_datetime(ts.min(), unit=ts_unit), pd.to_datetime(ts.max(), unit=ts_unit)

dataset_overview = {
    "total_interactions": int(len(events_raw)),
    "unique_users": int(events_raw["visitorid"].nunique()),
    "unique_products": int(events_raw["itemid"].nunique()),
    "first_event": str(t_min),
    "last_event": str(t_max),
    "days_covered": int((t_max - t_min).days),
    "timestamp_unit": ts_unit,
}
for k, v in dataset_overview.items():
    print(f"{k:<20}: {v:,}" if isinstance(v, int) else f"{k:<20}: {v}")

# Conversion funnel: how many users ever performed each event type
funnel = events_raw.groupby("event")["visitorid"].nunique().reindex(["view", "addtocart", "transaction"])
print("\nUsers who performed each event type:")
display(funnel.to_frame("unique_users"))

In [ ]:
# ---------------------------------------------------------------
# Inspection plots
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

# (a) Event type distribution (log scale, since views dominate)
sns.barplot(x=event_dist.index, y=event_dist["count"], ax=axes[0], palette="viridis")
axes[0].set_yscale("log")
axes[0].set_title("Interaction type distribution (log scale)")
axes[0].set_xlabel("event"); axes[0].set_ylabel("count")

# (b) Events per day
daily = pd.to_datetime(events_raw["timestamp"], unit=ts_unit).dt.floor("D").value_counts().sort_index()
axes[1].plot(daily.index, daily.values, lw=1)
axes[1].set_title("Events per day"); axes[1].set_ylabel("events")
axes[1].tick_params(axis="x", rotation=30)

# (c) User activity (long tail): events per user, log-log histogram
per_user = events_raw.groupby("visitorid").size()
bins = np.logspace(0, np.log10(per_user.max() + 1), 40)
axes[2].hist(per_user, bins=bins, color="#4C72B0")
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_title("Events per user (log-log)")
axes[2].set_xlabel("events per user"); axes[2].set_ylabel("number of users")

plt.tight_layout()
save_fig("01_dataset_inspection.png")

print(f"Users with exactly 1 event : {(per_user == 1).mean()*100:.1f}%")
print(f"Median events per user     : {per_user.median():.0f}")
print(f"Max events by a single user: {per_user.max():,}")

---
# 4. Preprocessing

### 4.1 Interaction weights: why View = 1, Add to cart = 3, Transaction = 5
The dataset has **implicit feedback**: no star ratings, only actions. Each action reveals a different level of interest:

| Event | Weight | Reasoning |
|---|---|---|
| `view` | **1** | Weakest signal. Users browse many products they never buy, and views can be accidental or exploratory. |
| `addtocart` | **3** | Clear purchase intent. It takes deliberate effort, and far fewer users do it than view. |
| `transaction` | **5** | Strongest signal. The user paid, so this is real preference. |

The weights follow how **rare** and **costly** each action is: in Retailrocket, views are the large majority of events, while add-to-cart and transactions are a few percent (see Section 3). A rarer, more deliberate action should count for more. We use a moderate scale (1/3/5) instead of extreme ratios so that purchases matter more without drowning out browsing, which still holds most of the information.

When a user interacts with the same product several times, the weights are **summed** (view + add to cart + buy = 9), then **capped at `MAX_EDGE_WEIGHT` = 15** so that one user refreshing a page 200 times cannot dominate.

### 4.2 Cleaning
- Normalise event names and keep only the three known types
- Drop rows with missing key fields
- Remove duplicate events (same user, product, event and millisecond)
- Convert Unix-millisecond timestamps to datetimes

In [ ]:
# ---------------------------------------------------------------
# Cleaning + weighting
# ---------------------------------------------------------------
events = events_raw[REQUIRED_COLUMNS].copy()
n0 = len(events)

events["event"] = events["event"].astype(str).str.strip().str.lower()
unknown_mask = ~events["event"].isin(EVENT_WEIGHTS.keys())
if unknown_mask.any():
    print(f"Dropping {unknown_mask.sum():,} rows with unknown event types: {events.loc[unknown_mask,'event'].unique()[:5]}")
events = events[~unknown_mask]

events = events.dropna(subset=REQUIRED_COLUMNS)
n_after_na = len(events)
events = events.drop_duplicates(subset=REQUIRED_COLUMNS)
n_after_dupes = len(events)

# Compact dtypes (saves memory on 2.7M rows)
events["visitorid"] = events["visitorid"].astype(np.int64)
events["itemid"] = events["itemid"].astype(np.int64)
events["timestamp"] = events["timestamp"].astype(np.int64)
if ts_unit == "s":                            # store everything in milliseconds
    events["timestamp"] = events["timestamp"] * 1000
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")
events["weight"] = events["event"].map(EVENT_WEIGHTS).astype(np.int32)
events["event"] = events["event"].astype("category")
events = events.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

cleaning_log = {
    "raw_rows": n0,
    "removed_unknown_or_missing": n0 - n_after_na,
    "removed_duplicates": n_after_na - n_after_dupes,
    "clean_rows": len(events),
}
print(json.dumps(cleaning_log, indent=2))
display(events.head())

### 4.3 Chronological train-test split (no data leakage)
A random split would let the model "see the future": it could train on a purchase from September and then be tested on predicting a view from June. That inflates scores.

Instead we sort all events by time and cut once:
- **Train** = events **before** the cutoff (the earliest `TRAIN_FRACTION` = 80% of events)
- **Test** = events **on or after** the cutoff

This matches real deployment: we learn from the past and predict what users will do next.

**Leakage rules applied throughout:**
1. Filtering thresholds, popularity, the similarity matrix and the graph all come from **train only**.
2. A test item counts as relevant only if the user did **not** already interact with it in train. Recommenders exclude seen items, so a re-view could never count as a hit.
3. Test items absent from train cannot be recommended by any of the methods (item cold start), so they are excluded from ground truth for **all** methods equally.

In [ ]:
# ---------------------------------------------------------------
# Chronological split
# ---------------------------------------------------------------
CUTOFF_TS = int(events["timestamp"].quantile(TRAIN_FRACTION))
train_events = events[events["timestamp"] < CUTOFF_TS].copy()
test_events = events[events["timestamp"] >= CUTOFF_TS].copy()

print(f"Cutoff time  : {pd.to_datetime(CUTOFF_TS, unit='ms')}")
print(f"Train events : {len(train_events):,}  ({train_events['datetime'].min()} -> {train_events['datetime'].max()})")
print(f"Test events  : {len(test_events):,}  ({test_events['datetime'].min()} -> {test_events['datetime'].max()})")
assert train_events["timestamp"].max() < test_events["timestamp"].min(), "Train and test overlap in time!"

### 4.4 Aggregation and filtering (train only)
- **Aggregation:** many raw events become **one row per (user, product)**, holding the summed and capped weight. This row becomes one edge in the graph and one cell in the matrix.
- **Bot removal:** visitors who touched more than `MAX_USER_ITEMS` distinct products look like crawlers. They would become huge hubs in the graph and distort Common Neighbors and PageRank.
- **Iterative k-core filtering:** keep users with ≥ `MIN_USER_ITEMS` products and products with ≥ `MIN_ITEM_USERS` users. Removing a product can push a user below the threshold, so this repeats until nothing changes. Most Retailrocket visitors have one or two events, and nobody can make personalised recommendations from a single view.

These functions are reused in the scalability experiments.

In [ ]:
# ---------------------------------------------------------------
# Reusable preprocessing functions
# ---------------------------------------------------------------
def aggregate_interactions(ev, max_weight=MAX_EDGE_WEIGHT):
    """Collapse raw events into one row per (visitorid, itemid) with summed and capped weight."""
    agg = (ev.groupby(["visitorid", "itemid"], observed=True)
             .agg(raw_weight=("weight", "sum"),
                  n_events=("weight", "size"),
                  max_event_weight=("weight", "max"),
                  last_timestamp=("timestamp", "max"))
             .reset_index())
    agg["weight"] = agg["raw_weight"].clip(upper=max_weight).astype(np.float32)
    return agg


def filter_interactions(inter, min_user_items=MIN_USER_ITEMS, min_item_users=MIN_ITEM_USERS,
                        max_user_items=MAX_USER_ITEMS, max_rounds=50, verbose=True):
    """Remove bot-like users, then iteratively enforce minimum user/item activity (k-core style)."""
    log = []
    user_deg = inter.groupby("visitorid").size()
    bots = user_deg[user_deg > max_user_items].index
    inter = inter[~inter["visitorid"].isin(bots)]
    log.append(f"removed {len(bots):,} bot-like users (> {max_user_items} products)")
    for r in range(1, max_rounds + 1):
        n_before = len(inter)
        item_deg = inter.groupby("itemid")["visitorid"].nunique()
        inter = inter[inter["itemid"].isin(item_deg[item_deg >= min_item_users].index)]
        user_deg = inter.groupby("visitorid")["itemid"].nunique()
        inter = inter[inter["visitorid"].isin(user_deg[user_deg >= min_user_items].index)]
        if len(inter) == n_before:
            log.append(f"converged after {r} rounds")
            break
    else:
        log.append(f"stopped after {max_rounds} rounds (not fully converged)")
    if verbose:
        for line in log:
            print("  -", line)
    return inter.reset_index(drop=True)


def build_ground_truth(test_ev, train_inter):
    """Relevant items per user = test items that are in the train catalogue and new to that user."""
    train_users = set(train_inter["visitorid"].unique())
    train_items = set(train_inter["itemid"].unique())
    seen = train_inter.groupby("visitorid")["itemid"].agg(set)

    t = test_ev[test_ev["visitorid"].isin(train_users) & test_ev["itemid"].isin(train_items)]
    gt = {}
    for u, items in t.groupby("visitorid")["itemid"]:
        rel = set(items.unique()) - seen.get(u, set())
        if rel:
            gt[u] = rel
    stats = {
        "test_users_total": int(test_ev["visitorid"].nunique()),
        "test_users_known_in_train": int(test_ev.loc[test_ev["visitorid"].isin(train_users), "visitorid"].nunique()),
        "test_users_with_new_relevant_items": len(gt),
        "avg_relevant_items_per_user": float(np.mean([len(v) for v in gt.values()])) if gt else 0.0,
    }
    return gt, stats

In [ ]:
# ---------------------------------------------------------------
# Build the clean training interaction dataset
# ---------------------------------------------------------------
t0 = time.perf_counter()
train_all = aggregate_interactions(train_events)
print(f"Aggregated train: {len(train_all):,} user-product pairs "
      f"({train_all['visitorid'].nunique():,} users, {train_all['itemid'].nunique():,} products)")

print("Filtering:")
train_interactions = filter_interactions(train_all)
print(f"Preprocessing time: {time.perf_counter()-t0:.1f}s")

if len(train_interactions) == 0:
    raise RuntimeError("Filtering removed everything. Lower MIN_USER_ITEMS / MIN_ITEM_USERS in the config cell.")

ground_truth, gt_stats = build_ground_truth(test_events, train_interactions)
if len(ground_truth) == 0:
    raise RuntimeError("No test users have new relevant items. Lower the filtering thresholds or TRAIN_FRACTION.")

n_users = train_interactions["visitorid"].nunique()
n_items = train_interactions["itemid"].nunique()
clean_summary = {
    "train_users": int(n_users),
    "train_products": int(n_items),
    "train_interactions": int(len(train_interactions)),
    "matrix_sparsity_pct": round(100 * (1 - len(train_interactions) / (n_users * n_items)), 4),
    "avg_products_per_user": round(len(train_interactions) / n_users, 2),
    "avg_users_per_product": round(len(train_interactions) / n_items, 2),
    **gt_stats,
}
print(json.dumps(clean_summary, indent=2))
print("\nEvent-strength mix in the filtered training pairs (max event per pair):")
print(train_interactions["max_event_weight"].map({1: "view", 3: "addtocart", 5: "transaction"}).value_counts())
display(train_interactions.head())

> **Note for the viva:** many Retailrocket visitors never come back after the cutoff, or come back only to look at products they have already seen. That is why the number of evaluable test users is much smaller than the total number of visitors. It is a property of the data (session-based browsing), not a bug.

---
# 5. Baseline: Item-Item Collaborative Filtering

### How it works, in simple terms
"People who interacted with this product also interacted with that one."

1. Build the **user-item matrix** $R$: rows are users, columns are products, and each cell holds the interaction weight.
2. Treat each **product as a vector** of user interactions (a column of $R$).
3. **Cosine similarity** between two products is the cosine of the angle between their vectors:
$$\text{sim}(i,j) = \frac{\vec{r}_i \cdot \vec{r}_j}{\lVert \vec{r}_i\rVert \, \lVert \vec{r}_j\rVert}$$
   It is 1 when the same users interacted with both in the same proportions, and 0 when no user touched both.
4. A user's **score** for product $j$ = $\sum_{i \in \text{history}} w_{u,i} \cdot \text{sim}(i,j)$: products similar to what they liked, weighted by how strongly they liked it.
5. Remove products already seen and return the top 5.

**Scalability trick:** a full item × item similarity matrix for tens of thousands of products would not fit in memory. We compute it **in chunks** and keep only the `CF_NEIGHBORS` = 50 most similar items per product (k-nearest-neighbour item similarity).

### Toy example
Four users and five products:

In [ ]:
# ---------------------------------------------------------------
# Toy example (worked by hand, then reused as a unit test in Section 10)
# ---------------------------------------------------------------
toy_names = {101: "Phone", 102: "Phone case", 103: "Charger", 104: "Laptop", 105: "Mouse"}
toy_users = {1: "Asha", 2: "Ben", 3: "Chen", 4: "Dev"}
toy_interactions = pd.DataFrame([
    (1, 101, 5), (1, 102, 3),                 # Asha bought a phone, carted a case
    (2, 101, 1), (2, 102, 1), (2, 103, 3),    # Ben viewed phone + case, carted a charger
    (3, 104, 5), (3, 105, 3),                 # Chen: laptop + mouse
    (4, 101, 3), (4, 103, 1), (4, 105, 1),    # Dev: phone, charger, mouse
], columns=["visitorid", "itemid", "weight"]).astype({"weight": np.float32})

R_toy = toy_interactions.pivot_table(index="visitorid", columns="itemid", values="weight", fill_value=0)
R_toy_named = R_toy.rename(index=toy_users, columns=toy_names)
print("User-item matrix (weights):")
display(R_toy_named)

S_toy = pd.DataFrame(cosine_similarity(R_toy.T.values), index=R_toy_named.columns, columns=R_toy_named.columns)
print("Item-item cosine similarity:")
display(S_toy.round(3))

asha = R_toy.loc[1].values
scores = pd.Series(asha @ S_toy.values, index=R_toy_named.columns)
scores[asha > 0] = np.nan          # exclude what Asha already has
print("Asha's scores (seen products excluded):")
display(scores.sort_values(ascending=False).round(3).to_frame("score"))
print("-> The Charger ranks first: it shares users with the Phone and Phone case Asha already likes.")

In [ ]:
# ---------------------------------------------------------------
# Shared helper: popularity fallback (used by every recommender)
# ---------------------------------------------------------------
def popularity_ranking(interactions):
    """Products ordered by number of distinct users in TRAIN (ties broken by itemid for determinism)."""
    pop = interactions.groupby("itemid")["visitorid"].nunique().reset_index(name="n_users")
    pop = pop.sort_values(["n_users", "itemid"], ascending=[False, True])
    return pop["itemid"].to_numpy()


def fill_with_popular(recs, popular_items, exclude, k):
    """Top up a recommendation list with popular unseen items until it has k entries."""
    have = {i for i, _ in recs} | set(exclude)
    for item in popular_items:
        if len(recs) >= k:
            break
        if item not in have:
            recs.append((int(item), 0.0))
            have.add(item)
    return recs


class PopularityRecommender:
    """Non-personalised reference: recommend the most popular unseen products."""
    name = "Popularity (reference)"

    def __init__(self, interactions):
        self.popular_items = popularity_ranking(interactions)
        self.seen = interactions.groupby("visitorid")["itemid"].agg(set).to_dict()
        self.last_info = {}

    def recommend(self, user_id, k=TOP_K):
        seen = self.seen.get(user_id, set())
        self.last_info = {"fallback": False, "n_padded": 0}
        return fill_with_popular([], self.popular_items, seen, k)

In [ ]:
# ---------------------------------------------------------------
# Item-Item Collaborative Filtering recommender
# ---------------------------------------------------------------
class ItemCFRecommender:
    """Item-based CF with cosine similarity on the weighted user-item matrix (top-N neighbours per item)."""
    name = "Item-Item CF"

    def __init__(self, n_neighbors=CF_NEIGHBORS, chunk_size=1000, min_history=MIN_HISTORY):
        self.n_neighbors = n_neighbors
        self.chunk_size = chunk_size
        self.min_history = min_history
        self.last_info = {}

    def fit(self, interactions):
        # 1) index maps: real IDs <-> matrix positions
        self.user_ids = np.sort(interactions["visitorid"].unique())
        self.item_ids = np.sort(interactions["itemid"].unique())
        self.user_index = {u: i for i, u in enumerate(self.user_ids)}
        self.item_index = {it: j for j, it in enumerate(self.item_ids)}
        n_u, n_i = len(self.user_ids), len(self.item_ids)

        # 2) sparse user-item matrix R (users x items)
        rows = interactions["visitorid"].map(self.user_index).to_numpy()
        cols = interactions["itemid"].map(self.item_index).to_numpy()
        vals = interactions["weight"].to_numpy(dtype=np.float32)
        self.R = sp.csr_matrix((vals, (rows, cols)), shape=(n_u, n_i), dtype=np.float32)

        # 3) cosine similarity = dot product of L2-normalised item columns, computed chunk by chunk
        Rn = normalize(self.R.tocsc(), norm="l2", axis=0).tocsr()   # users x items, unit-length columns
        Rn_T = Rn.T.tocsr()                                         # items x users
        k = min(self.n_neighbors, n_i - 1)
        data, indices, indptr = [], [], [0]
        for start in range(0, n_i, self.chunk_size):
            end = min(start + self.chunk_size, n_i)
            block = (Rn_T[start:end] @ Rn).toarray()                # (chunk x items) cosine similarities
            block[np.arange(end - start), np.arange(start, end)] = 0.0   # no self-similarity
            if k > 0:
                top = np.argpartition(-block, k - 1, axis=1)[:, :k]
                top_vals = np.take_along_axis(block, top, axis=1)
            for r in range(end - start):
                if k > 0:
                    keep = top_vals[r] > 0
                    order = np.argsort(top[r][keep])
                    indices.extend(top[r][keep][order]); data.extend(top_vals[r][keep][order])
                indptr.append(len(indices))
        self.S = sp.csr_matrix((np.array(data, dtype=np.float32), np.array(indices, dtype=np.int32),
                                np.array(indptr, dtype=np.int64)), shape=(n_i, n_i))
        self.popular_items = popularity_ranking(interactions)
        return self

    def similarity_matrix_mb(self):
        return (self.S.data.nbytes + self.S.indices.nbytes + self.S.indptr.nbytes) / 1e6

    def recommend(self, user_id, k=TOP_K):
        self.last_info = {"fallback": False, "n_padded": 0}
        if user_id not in self.user_index:
            self.last_info["fallback"] = True
            return fill_with_popular([], self.popular_items, set(), k)
        u_row = self.R[self.user_index[user_id]]
        seen_idx = u_row.indices
        seen_items = set(self.item_ids[seen_idx])
        if len(seen_idx) < self.min_history:            # insufficient history -> popularity
            self.last_info["fallback"] = True
            return fill_with_popular([], self.popular_items, seen_items, k)

        scores = np.asarray((u_row @ self.S).todense()).ravel()   # sum_i w_ui * sim(i, j)
        scores[seen_idx] = -np.inf                                  # exclude seen products
        n_pos = int((scores > 0).sum())
        take = min(k, n_pos)
        recs = []
        if take > 0:
            top = np.argpartition(-scores, take - 1)[:take]
            top = top[np.argsort(-scores[top], kind="stable")]
            recs = [(int(self.item_ids[j]), float(scores[j])) for j in top]
        if len(recs) < k:
            self.last_info["n_padded"] = k - len(recs)
            recs = fill_with_popular(recs, self.popular_items, seen_items, k)
        return recs

In [ ]:
# ---------------------------------------------------------------
# Fit the baseline on the training data
# ---------------------------------------------------------------
gc.collect()
mem0 = rss_mb()
t0 = time.perf_counter()
cf_model = ItemCFRecommender().fit(train_interactions)
CF_FIT_TIME = time.perf_counter() - t0
print(f"User-item matrix : {cf_model.R.shape[0]:,} users x {cf_model.R.shape[1]:,} products, {cf_model.R.nnz:,} non-zeros")
print(f"Item similarity  : {cf_model.S.nnz:,} stored neighbour pairs ({cf_model.similarity_matrix_mb():.1f} MB)")
print(f"Fit time         : {CF_FIT_TIME:.2f}s   |   memory change: {rss_mb()-mem0:+.0f} MB")

pop_model = PopularityRecommender(train_interactions)

# Demo user: a reproducibly chosen evaluation user with a mid-sized history
rng_demo = np.random.default_rng(SEED)
gt_users_sorted = sorted(ground_truth.keys())
hist_sizes = train_interactions.groupby("visitorid").size()
candidates = [u for u in gt_users_sorted if 5 <= hist_sizes.get(u, 0) <= 30] or gt_users_sorted
DEMO_USER = int(rng_demo.choice(candidates))
print(f"\nDemo user: {DEMO_USER} | train history: {hist_sizes[DEMO_USER]} products")
display(pd.DataFrame(cf_model.recommend(DEMO_USER), columns=["itemid", "cf_score"]))

---
# 6. Graph Construction (NetworkX)
We model the data as a **weighted bipartite graph** $G = (U \cup P, E)$:
- **User nodes** (`U_<visitorid>`) and **product nodes** (`P_<itemid>`) form two separate sets
- An **edge** (user, product) exists if the user interacted with the product in train. Its `weight` is the aggregated interaction weight
- Edges only connect a user to a product, never user-user or product-product. That is what *bipartite* means.

Bipartite density is $\frac{|E|}{|U|\cdot|P|}$, the fraction of all possible user-product links that actually exist.

In [ ]:
# ---------------------------------------------------------------
# Graph builder + statistics (reused in scalability experiments)
# ---------------------------------------------------------------
def user_node(uid):   return f"U_{uid}"
def product_node(iid): return f"P_{iid}"
def node_id(node):    return int(node[2:])


def build_bipartite_graph(interactions):
    """Weighted, undirected bipartite user-product graph."""
    G = nx.Graph()
    u_nodes = ("U_" + interactions["visitorid"].astype(str)).to_numpy()
    p_nodes = ("P_" + interactions["itemid"].astype(str)).to_numpy()
    G.add_nodes_from(pd.unique(u_nodes), bipartite=0, node_type="user")
    G.add_nodes_from(pd.unique(p_nodes), bipartite=1, node_type="product")
    G.add_weighted_edges_from(zip(u_nodes, p_nodes, interactions["weight"].astype(float).to_numpy()))
    return G


def graph_statistics(G):
    users = [n for n, d in G.nodes(data=True) if d["node_type"] == "user"]
    prods = [n for n, d in G.nodes(data=True) if d["node_type"] == "product"]
    u_deg = np.array([G.degree(n) for n in users])
    p_deg = np.array([G.degree(n) for n in prods])
    comps = list(nx.connected_components(G))
    largest = max(len(c) for c in comps) if comps else 0
    w = np.array([d["weight"] for _, _, d in G.edges(data=True)])
    return {
        "user_nodes": len(users),
        "product_nodes": len(prods),
        "total_nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "bipartite_density": G.number_of_edges() / (len(users) * len(prods)) if users and prods else 0.0,
        "networkx_density": nx.density(G),
        "avg_user_degree": float(u_deg.mean()), "median_user_degree": float(np.median(u_deg)), "max_user_degree": int(u_deg.max()),
        "avg_product_degree": float(p_deg.mean()), "median_product_degree": float(np.median(p_deg)), "max_product_degree": int(p_deg.max()),
        "connected_components": len(comps),
        "largest_component_nodes": largest,
        "largest_component_pct": 100 * largest / G.number_of_nodes(),
        "avg_edge_weight": float(w.mean()),
        "is_bipartite": nx.is_bipartite(G),
    }, u_deg, p_deg

In [ ]:
# ---------------------------------------------------------------
# Build the training graph
# ---------------------------------------------------------------
gc.collect()
mem0 = rss_mb()
t0 = time.perf_counter()
G = build_bipartite_graph(train_interactions)
GRAPH_BUILD_TIME = time.perf_counter() - t0
GRAPH_MEM_MB = rss_mb() - mem0

graph_stats, user_degrees, product_degrees = graph_statistics(G)
graph_stats["build_time_s"] = GRAPH_BUILD_TIME
graph_stats["build_memory_mb"] = GRAPH_MEM_MB

display(pd.Series(graph_stats, name="value").to_frame())

In [ ]:
# ---------------------------------------------------------------
# Degree distributions
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, deg, label, color in [(axes[0], user_degrees, "User", "#4C72B0"), (axes[1], product_degrees, "Product", "#DD8452")]:
    counts = Counter(deg)
    x, y = zip(*sorted(counts.items()))
    ax.scatter(x, y, s=12, color=color, alpha=0.8)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"{label} degree distribution (log-log)")
    ax.set_xlabel("degree (number of connected nodes)"); ax.set_ylabel("number of nodes")
plt.tight_layout()
save_fig("02_degree_distribution.png")
print("A roughly straight line on log-log axes means a heavy-tailed (power-law-like) distribution:")
print("a few hub products and users have very many connections, and most have few.")

### Visualising a manageable subgraph
The full graph has tens of thousands of nodes, which cannot be drawn usefully. We draw the **2-hop neighbourhood** of the demo user instead: their products (strongest first) and a few other users of each product. Paths through this neighbourhood are what Common Neighbors and PageRank follow.

In [ ]:
# ---------------------------------------------------------------
# Subgraph extraction + visualisation (matplotlib + interactive Plotly)
# ---------------------------------------------------------------
def extract_subgraph(G, center, max_items=8, max_users_per_item=4):
    """Center user + their strongest products + top co-users of each product."""
    items = sorted(G[center], key=lambda p: -G[center][p]["weight"])[:max_items]
    nodes = {center, *items}
    for p in items:
        others = sorted((u for u in G[p] if u != center), key=lambda u: -G[p][u]["weight"])[:max_users_per_item]
        nodes.update(others)
    return G.subgraph(nodes).copy()


def plotly_subgraph(H, center, title):
    pos = nx.spring_layout(H, seed=SEED, weight="weight")
    edge_x, edge_y = [], []
    for a, b in H.edges():
        edge_x += [pos[a][0], pos[b][0], None]; edge_y += [pos[a][1], pos[b][1], None]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="#BBBBBB"),
                             hoverinfo="none", showlegend=False))
    for ntype, color, symbol in [("user", "#4C72B0", "circle"), ("product", "#DD8452", "square")]:
        ns = [n for n in H if H.nodes[n]["node_type"] == ntype]
        fig.add_trace(go.Scatter(
            x=[pos[n][0] for n in ns], y=[pos[n][1] for n in ns], mode="markers+text",
            marker=dict(size=[22 if n == center else 13 for n in ns], color=color, symbol=symbol,
                        line=dict(width=2, color="black")),
            text=[n for n in ns], textposition="top center", textfont=dict(size=9),
            hovertext=[f"{n}<br>degree in full graph: {G.degree(n)}" for n in ns], hoverinfo="text",
            name=ntype.capitalize() + "s"))
    fig.update_layout(title=title, xaxis=dict(visible=False), yaxis=dict(visible=False),
                      height=600, plot_bgcolor="white")
    return fig


DEMO_NODE = user_node(DEMO_USER)
H = extract_subgraph(G, DEMO_NODE)
print(f"Subgraph: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

plt.figure(figsize=(11, 7))
pos = nx.spring_layout(H, seed=SEED, weight="weight")
colors = ["#C44E52" if n == DEMO_NODE else ("#4C72B0" if H.nodes[n]["node_type"] == "user" else "#DD8452") for n in H]
widths = [0.5 + H[a][b]["weight"] / 3 for a, b in H.edges()]
nx.draw_networkx(H, pos, node_color=colors, node_size=420, font_size=7, width=widths, edge_color="#999999")
plt.title(f"Bipartite subgraph around {DEMO_NODE} (red). Blue = users, orange = products, edge width = weight")
plt.axis("off")
save_fig("03_subgraph_demo_user.png")

save_plotly(plotly_subgraph(H, DEMO_NODE, f"Interactive bipartite subgraph around {DEMO_NODE}"),
            "03_subgraph_demo_user.html")

---
# 7. Graph-Based Recommendation Algorithms

### A. Common Neighbors (user-based, through the graph)
In the bipartite graph, two users are linked through every product they share. The **number of common product neighbours** measures how similar they are:
$$CN(u, v) = |N(u) \cap N(v)|$$
**Steps:**
1. From user $u$, walk to each of their products, then to every other user of those products, counting how often each user is reached. That count is $CN(u,v)$.
2. Keep the `CN_TOP_USERS` = 50 most similar users.
3. Score each product $j$ that $u$ has not seen: $\text{score}(j) = \sum_{v} CN(u,v)\cdot w_{v,j}$. A product counts for more when similar users used it and interacted with it strongly.
4. Return the top 5.

*Intuition:* "Find people whose basket looks like yours and show what else they liked."

### B. Personalized PageRank (PPR)
Picture a random surfer on the graph:
- At each step, with probability $\alpha$ = 0.85 they follow an edge (choosing heavier edges more often), and
- with probability $1-\alpha$ = 0.15 they **jump back to the target user** (restart).

Over many steps, the fraction of time spent on each node is its PPR score. Products the surfer visits often are well connected to the user through **many paths of any length**, not only direct 2-hop overlaps. Because of the restart, the scores stay **personalised** to that user.

$$\vec{\pi} = \alpha\, \vec{\pi} M + (1-\alpha)\, \vec{e}_u$$
($M$ is the weight-normalised transition matrix and $\vec{e}_u$ puts all restart probability on user $u$.)

We call `networkx.pagerank(G, alpha, personalization={user: 1}, weight="weight")`, keep only product nodes, remove seen products, and return the top 5.

**CN vs PPR:** CN only looks 2 hops away (user → product → user) and then one more hop to candidates. PPR follows paths of all lengths with decreasing weight, so it can surface products from the wider neighbourhood. It also has to touch the whole graph for every user, which makes it slower (see Section 8).

In [ ]:
# ---------------------------------------------------------------
# Common Neighbors recommender
# ---------------------------------------------------------------
class CommonNeighborsRecommender:
    name = "Common Neighbors"

    def __init__(self, G, popular_items, top_users=CN_TOP_USERS, min_history=MIN_HISTORY):
        self.G, self.popular_items = G, popular_items
        self.top_users, self.min_history = top_users, min_history
        self.last_info = {}

    def similar_users(self, unode):
        """CN(u, v) for every user v that shares at least one product with u."""
        cn = Counter()
        for p in self.G[unode]:
            for v in self.G[p]:
                if v != unode:
                    cn[v] += 1
        # sort by count desc, then node id for deterministic ties
        return sorted(cn.items(), key=lambda x: (-x[1], x[0]))[: self.top_users]

    def recommend(self, user_id, k=TOP_K):
        self.last_info = {"fallback": False, "n_padded": 0}
        unode = user_node(user_id)
        if unode not in self.G or self.G.degree(unode) < self.min_history:
            self.last_info["fallback"] = True
            seen = {node_id(p) for p in self.G[unode]} if unode in self.G else set()
            return fill_with_popular([], self.popular_items, seen, k)
        seen_nodes = set(self.G[unode])
        scores = defaultdict(float)
        for v, c in self.similar_users(unode):
            for p, data in self.G[v].items():
                if p not in seen_nodes:
                    scores[p] += c * data["weight"]
        top = heapq.nsmallest(k, scores.items(), key=lambda x: (-x[1], x[0]))
        recs = [(node_id(p), float(s)) for p, s in top]
        if len(recs) < k:
            self.last_info["n_padded"] = k - len(recs)
            recs = fill_with_popular(recs, self.popular_items, {node_id(p) for p in seen_nodes}, k)
        return recs

In [ ]:
# ---------------------------------------------------------------
# Personalized PageRank recommender
# ---------------------------------------------------------------
class PersonalizedPageRankRecommender:
    name = "Personalized PageRank"

    def __init__(self, G, popular_items, alpha=PPR_ALPHA, max_iter=PPR_MAX_ITER, tol=PPR_TOL,
                 min_history=MIN_HISTORY):
        self.G, self.popular_items = G, popular_items
        self.alpha, self.max_iter, self.tol, self.min_history = alpha, max_iter, tol, min_history
        self.product_nodes = [n for n, d in G.nodes(data=True) if d["node_type"] == "product"]
        self.last_info = {}

    def scores_for(self, unode):
        """Full PPR vector personalised on one user node (NetworkX implementation)."""
        return nx.pagerank(self.G, alpha=self.alpha, personalization={unode: 1.0},
                           weight="weight", max_iter=self.max_iter, tol=self.tol)

    def recommend(self, user_id, k=TOP_K):
        self.last_info = {"fallback": False, "n_padded": 0}
        unode = user_node(user_id)
        if unode not in self.G or self.G.degree(unode) < self.min_history:
            self.last_info["fallback"] = True
            seen = {node_id(p) for p in self.G[unode]} if unode in self.G else set()
            return fill_with_popular([], self.popular_items, seen, k)
        seen_nodes = set(self.G[unode])
        try:
            pr = self.scores_for(unode)
        except nx.PowerIterationFailedConvergence:
            print(f"PPR did not converge for user {user_id}; using popularity fallback.")
            self.last_info["fallback"] = True
            return fill_with_popular([], self.popular_items, {node_id(p) for p in seen_nodes}, k)
        cand = ((pr[p], p) for p in self.product_nodes if p not in seen_nodes and pr[p] > 0)
        top = heapq.nlargest(k, cand)
        recs = [(node_id(p), float(s)) for s, p in top]
        if len(recs) < k:
            self.last_info["n_padded"] = k - len(recs)
            recs = fill_with_popular(recs, self.popular_items, {node_id(p) for p in seen_nodes}, k)
        return recs

In [ ]:
# ---------------------------------------------------------------
# Toy example: run all three real implementations on the 4-user toy data from Section 5
# ---------------------------------------------------------------
toy_pop = popularity_ranking(toy_interactions)
toy_G = build_bipartite_graph(toy_interactions)
toy_models = [ItemCFRecommender(n_neighbors=10).fit(toy_interactions),
              CommonNeighborsRecommender(toy_G, toy_pop),
              PersonalizedPageRankRecommender(toy_G, toy_pop)]
rows = []
for m in toy_models:
    for rank, (item, score) in enumerate(m.recommend(1, k=3), 1):
        rows.append({"method": m.name, "rank": rank, "product": toy_names[item], "score": round(score, 4)})
print("Recommendations for Asha (already has Phone + Phone case):")
display(pd.DataFrame(rows))
print("CN detail: Asha shares 2 products with Ben and 1 with Dev, so CN(Asha,Ben)=2 and CN(Asha,Dev)=1.")
print("Charger score = 2*3 (Ben carted it) + 1*1 (Dev viewed it) = 7; Mouse score = 1*1 (Dev) = 1.")

In [ ]:
# ---------------------------------------------------------------
# Instantiate on the real training graph and compare for the demo user
# ---------------------------------------------------------------
popular_items = popularity_ranking(train_interactions)
cn_model = CommonNeighborsRecommender(G, popular_items)
ppr_model = PersonalizedPageRankRecommender(G, popular_items)

history = (train_interactions[train_interactions["visitorid"] == DEMO_USER]
           .sort_values("weight", ascending=False)[["itemid", "weight", "n_events", "max_event_weight"]])
print(f"Training history of demo user {DEMO_USER} ({len(history)} products):")
display(history.head(15))

demo_rows = []
for m in [cf_model, cn_model, ppr_model]:
    t0 = time.perf_counter()
    recs = m.recommend(DEMO_USER)
    dt = (time.perf_counter() - t0) * 1000
    for rank, (item, score) in enumerate(recs, 1):
        demo_rows.append({"method": m.name, "rank": rank, "itemid": item, "score": score,
                          "in_test_ground_truth": item in ground_truth.get(DEMO_USER, set()),
                          "time_ms": round(dt, 1)})
demo_recs = pd.DataFrame(demo_rows)
display(demo_recs)
print("Products this user actually interacted with later (test):", sorted(ground_truth.get(DEMO_USER, set()))[:20])

---
# 8. Evaluation and Comparison

All methods are evaluated on the **same** chronological split and the **same** reproducible sample of test users. For each user $u$, with recommended list $L_u$ (5 items) and relevant set $T_u$ (new products they interacted with after the cutoff):

| Metric | Formula | Meaning |
|---|---|---|
| Precision@5 | $\frac{\lvert L_u \cap T_u\rvert}{5}$ | Share of the 5 recommendations that were correct |
| Recall@5 | $\frac{\lvert L_u \cap T_u\rvert}{\lvert T_u\rvert}$ | Share of the user's future products that we found |
| F1@5 | $\frac{2PR}{P+R}$ | Balance of precision and recall (0 when there are no hits) |
| NDCG@5 | $\frac{\sum_{i=1}^{5} rel_i/\log_2(i+1)}{\text{IDCG}}$ | Rewards correct items placed **higher** in the list (1 = perfect ranking) |
| Hit rate@5 | $\mathbb{1}[\lvert L_u \cap T_u\rvert>0]$ | Did we get at least one right? |

Metrics are averaged over users. We also report **95% bootstrap confidence intervals**: with a few hundred users the numbers are noisy, and the intervals show whether a difference is meaningful.

A **Popularity** recommender is included as a reference point only. A personalised method should beat "recommend the bestsellers".

> Absolute values are expected to be **low** (often a few percent). With tens of thousands of products and very sparse histories, guessing 5 exact future products is hard. What matters is the **relative** comparison.

In [ ]:
# ---------------------------------------------------------------
# Metric functions
# ---------------------------------------------------------------
def precision_at_k(recommended, relevant, k=TOP_K):
    return len(set(recommended[:k]) & relevant) / k

def recall_at_k(recommended, relevant, k=TOP_K):
    return len(set(recommended[:k]) & relevant) / len(relevant) if relevant else 0.0

def f1_at_k(recommended, relevant, k=TOP_K):
    p, r = precision_at_k(recommended, relevant, k), recall_at_k(recommended, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def ndcg_at_k(recommended, relevant, k=TOP_K):
    dcg = sum(1.0 / math.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

# Quick sanity check on a hand-made case
_rec, _rel = [10, 20, 30, 40, 50], {20, 99}
print("precision", precision_at_k(_rec, _rel), "| recall", recall_at_k(_rec, _rel),
      "| f1", round(f1_at_k(_rec, _rel), 4), "| ndcg", round(ndcg_at_k(_rec, _rel), 4))
# expected: precision 0.2, recall 0.5, f1 0.2857, ndcg = (1/log2(3)) / (1 + 1/log2(3)) = 0.3869

In [ ]:
# ---------------------------------------------------------------
# Evaluation loop (same users for every method)
# ---------------------------------------------------------------
def evaluate_recommender(model, eval_users, gt, k=TOP_K, progress_every=100):
    metric_rows, rec_rows = [], []
    for n, u in enumerate(eval_users, 1):
        t0 = time.perf_counter()
        recs = model.recommend(u, k)
        dt_ms = (time.perf_counter() - t0) * 1000
        items = [i for i, _ in recs]
        rel = gt[u]
        metric_rows.append({
            "visitorid": u, "method": model.name,
            "precision@5": precision_at_k(items, rel, k), "recall@5": recall_at_k(items, rel, k),
            "f1@5": f1_at_k(items, rel, k), "ndcg@5": ndcg_at_k(items, rel, k),
            "hit@5": float(len(set(items) & rel) > 0), "n_relevant": len(rel),
            "time_ms": dt_ms, "fallback": model.last_info.get("fallback", False),
            "n_padded": model.last_info.get("n_padded", 0)})
        for rank, (item, score) in enumerate(recs, 1):
            rec_rows.append({"visitorid": u, "method": model.name, "rank": rank, "itemid": item,
                             "score": score, "is_hit": item in rel})
        if progress_every and n % progress_every == 0:
            print(f"   {model.name}: {n}/{len(eval_users)} users")
    return pd.DataFrame(metric_rows), pd.DataFrame(rec_rows)


def bootstrap_ci(values, n_boot=N_BOOTSTRAP, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return tuple(np.percentile(means, [2.5, 97.5]))


rng_eval = np.random.default_rng(SEED)
all_gt_users = np.array(sorted(ground_truth.keys()))
EVAL_USERS = sorted(rng_eval.choice(all_gt_users, size=min(N_EVAL_USERS, len(all_gt_users)), replace=False).tolist())
print(f"Evaluating on {len(EVAL_USERS)} users (of {len(all_gt_users):,} with test ground truth)\n")

models = [cf_model, cn_model, ppr_model, pop_model]
eval_frames, rec_frames = [], []
for m in models:
    t0 = time.perf_counter()
    mdf, rdf = evaluate_recommender(m, EVAL_USERS, ground_truth)
    eval_frames.append(mdf); rec_frames.append(rdf)
    print(f"{m.name:<24} done in {time.perf_counter()-t0:.1f}s")

per_user_eval = pd.concat(eval_frames, ignore_index=True)
all_recommendations = pd.concat(rec_frames, ignore_index=True)

In [ ]:
# ---------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------
METRICS = ["precision@5", "recall@5", "f1@5", "ndcg@5", "hit@5"]
summary_rows = []
for name, g in per_user_eval.groupby("method", sort=False):
    row = {"method": name, "n_users": len(g)}
    for mtr in METRICS:
        row[mtr] = g[mtr].mean()
        lo, hi = bootstrap_ci(g[mtr])
        row[f"{mtr}_ci_low"], row[f"{mtr}_ci_high"] = lo, hi
    row["avg_time_ms"] = g["time_ms"].mean()
    row["median_time_ms"] = g["time_ms"].median()
    row["fallback_pct"] = 100 * g["fallback"].mean()
    row["padded_pct"] = 100 * (g["n_padded"] > 0).mean()
    summary_rows.append(row)
evaluation_summary = pd.DataFrame(summary_rows)

display_cols = ["method", "n_users"] + METRICS + ["avg_time_ms", "fallback_pct", "padded_pct"]
display(evaluation_summary[display_cols].style.format(
    {**{m: "{:.4f}" for m in METRICS}, "avg_time_ms": "{:.2f}", "fallback_pct": "{:.1f}", "padded_pct": "{:.1f}"}
).highlight_max(subset=METRICS, color="#c7e9c0"))

In [ ]:
# ---------------------------------------------------------------
# Metric comparison charts (static + interactive)
# ---------------------------------------------------------------
plot_metrics = ["precision@5", "recall@5", "f1@5", "ndcg@5"]
long = evaluation_summary.melt(id_vars="method", value_vars=plot_metrics, var_name="metric", value_name="value")
err_lo = evaluation_summary.melt(id_vars="method", value_vars=[f"{m}_ci_low" for m in plot_metrics], value_name="lo")["lo"]
err_hi = evaluation_summary.melt(id_vars="method", value_vars=[f"{m}_ci_high" for m in plot_metrics], value_name="hi")["hi"]
long["err_minus"] = long["value"] - err_lo.values
long["err_plus"] = err_hi.values - long["value"]

fig, ax = plt.subplots(figsize=(12, 5))
methods_order = evaluation_summary["method"].tolist()
width = 0.8 / len(methods_order)
xs = np.arange(len(plot_metrics))
for i, mname in enumerate(methods_order):
    sub = long[long["method"] == mname].set_index("metric").loc[plot_metrics]
    ax.bar(xs + i * width, sub["value"], width, label=mname, color=METHOD_COLORS.get(mname),
           yerr=[sub["err_minus"], sub["err_plus"]], capsize=3)
ax.set_xticks(xs + width * (len(methods_order) - 1) / 2)
ax.set_xticklabels(plot_metrics)
ax.set_ylabel("score (mean over users, 95% CI)")
ax.set_title("Recommendation quality @5")
ax.legend()
save_fig("04_metric_comparison.png")

fig = px.bar(long, x="metric", y="value", color="method", barmode="group",
             error_y="err_plus", error_y_minus="err_minus", color_discrete_map=METHOD_COLORS,
             title="Recommendation quality @5 (95% bootstrap CI)")
save_plotly(fig, "04_metric_comparison.html")

In [ ]:
# ---------------------------------------------------------------
# Execution time comparison
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
order = evaluation_summary.sort_values("avg_time_ms")["method"]
sns.barplot(data=evaluation_summary, x="method", y="avg_time_ms", order=order, ax=axes[0],
            palette=[METHOD_COLORS[m] for m in order])
axes[0].set_yscale("log"); axes[0].set_title("Average time per user (log scale)"); axes[0].set_ylabel("ms")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=per_user_eval, x="method", y="time_ms", order=order, ax=axes[1],
            palette=[METHOD_COLORS[m] for m in order], showfliers=False)
axes[1].set_yscale("log"); axes[1].set_title("Per-user time distribution"); axes[1].set_ylabel("ms")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
save_fig("05_execution_time.png")

fig = px.box(per_user_eval, x="method", y="time_ms", color="method", log_y=True, points=False,
             color_discrete_map=METHOD_COLORS, title="Recommendation time per user (ms, log scale)")
save_plotly(fig, "05_execution_time.html")

### Interpretation of the actual results
The cell below writes the interpretation **from the numbers just computed**, so it stays correct whatever your run produces. Use it and the charts above when you discuss results in the viva.

In [ ]:
# ---------------------------------------------------------------
# Data-driven interpretation (no hardcoded conclusions)
# ---------------------------------------------------------------
es = evaluation_summary.set_index("method")
personalised = ["Item-Item CF", "Common Neighbors", "Personalized PageRank"]
lines = []
for mtr in ["precision@5", "recall@5", "ndcg@5"]:
    best = es.loc[personalised, mtr].idxmax()
    lines.append(f"- Best {mtr} among personalised methods: **{best}** ({es.loc[best, mtr]:.4f}).")

base = es.loc["Item-Item CF", "ndcg@5"]
for m in ["Common Neighbors", "Personalized PageRank"]:
    v = es.loc[m, "ndcg@5"]
    rel = (v - base) / base * 100 if base > 0 else float("nan")
    overlap = not (es.loc[m, "ndcg@5_ci_low"] > es.loc["Item-Item CF", "ndcg@5_ci_high"] or
                   es.loc[m, "ndcg@5_ci_high"] < es.loc["Item-Item CF", "ndcg@5_ci_low"])
    lines.append(f"- {m} NDCG@5 vs Item-Item CF: {rel:+.1f}% "
                 + ("(confidence intervals overlap, so the difference may not be significant)." if overlap
                    else "(confidence intervals do not overlap, so the difference is likely real)."))

pop_ndcg = es.loc["Popularity (reference)", "ndcg@5"]
beats = [m for m in personalised if es.loc[m, "ndcg@5"] > pop_ndcg]
lines.append(f"- Methods beating the popularity reference on NDCG@5: {', '.join(beats) if beats else 'none'}.")

fastest = es.loc[personalised, "avg_time_ms"].idxmin(); slowest = es.loc[personalised, "avg_time_ms"].idxmax()
lines.append(f"- Fastest: **{fastest}** ({es.loc[fastest,'avg_time_ms']:.2f} ms/user); slowest: **{slowest}** "
             f"({es.loc[slowest,'avg_time_ms']:.2f} ms/user), {es.loc[slowest,'avg_time_ms']/es.loc[fastest,'avg_time_ms']:.0f}x slower.")
lines.append(f"- Hit rate@5: " + ", ".join(f"{m} {es.loc[m,'hit@5']*100:.1f}%" for m in personalised)
             + " of evaluated users got at least one correct recommendation.")
if (es.loc[personalised, "padded_pct"] > 0).any():
    lines.append("- Some lists were topped up with popular items because a method produced fewer than 5 candidates: "
                 + ", ".join(f"{m} {es.loc[m,'padded_pct']:.1f}%" for m in personalised) + ".")
INTERPRETATION = "\n".join(lines)

try:
    from IPython.display import Markdown
    display(Markdown(INTERPRETATION))
except ImportError:
    print(INTERPRETATION)

**General reasons behind typical patterns** (check them against your actual numbers above):
- **PPR is slowest** because NetworkX runs power iteration over the **entire graph** for every single user, so the cost grows with the number of edges. CF needs one sparse vector-matrix product. CN touches only the user's 2-hop neighbourhood.
- **Graph methods can beat CF** when useful signals are more than one hop away. PPR in particular uses multi-hop paths, which helps sparse users.
- **CF can beat graph methods** when item co-occurrence is strong and direct (for example "frequently bought together" accessories).
- **Popularity can be surprisingly strong** in e-commerce because demand is concentrated on a few hit products. A personalised method that barely beats it adds little value.

---
# 9. Scalability Analysis
We rerun the pipeline (aggregate → build CF → build graph → recommend → evaluate) on 10%, 30%, 60% and 100% of the users.

**Which users we sample from:** the users who passed filtering in Section 4 (the *active-user population*). An earlier version sampled from all 1.4M raw visitors and then re-ran filtering on each sample. That made small samples collapse: in a 10% sample every product has roughly a tenth of its users, so almost no product reaches the 5-user threshold, and the iterative filter removed everything. Sampling from the filtered population keeps each size comparable, and the 100% run reproduces the main pipeline exactly.

**Reproducible, nested sampling:** we shuffle those user IDs once with `SEED` and take the first 10%, 30%, ... of the list. Each smaller sample is contained in the larger one, and the same users are chosen on every run. We sample **users rather than raw events**, so every sampled user keeps their complete history; random event sampling would break up histories. The same global time cutoff is used at every size.

**Measured:** users, products, interactions, CF build time, graph construction time, memory (process RSS change while building the graph, measured with `psutil`), average recommendation time per user, and recommendation quality (on `N_SCALE_EVAL_USERS` users per size).

> Memory deltas from `psutil` are approximate. Python's allocator reuses freed memory, so treat the numbers as an order of magnitude.

In [ ]:
# ---------------------------------------------------------------
# Scalability experiment
# ---------------------------------------------------------------
def run_scalability(base_interactions, events_df, fractions=SCALE_FRACTIONS, cutoff_ts=CUTOFF_TS,
                    n_eval=N_SCALE_EVAL_USERS):
    """Nested user samples drawn from the filtered (active) user population."""
    rng = np.random.default_rng(SEED)
    pool = np.sort(base_interactions["visitorid"].unique())
    shuffled = rng.permutation(pool)
    results = []
    for frac in fractions:
        print(f"\n=== {int(frac*100)}% of active users ===")
        gc.collect()
        sample_users = set(shuffled[: max(1, int(round(frac * len(shuffled))))])
        ev = events_df[events_df["visitorid"].isin(sample_users)]
        tr_ev, te_ev = ev[ev["timestamp"] < cutoff_ts], ev[ev["timestamp"] >= cutoff_ts]

        # preprocessing cost on this sample's raw events (aggregation into weighted user-item pairs)
        t0 = time.perf_counter()
        _ = aggregate_interactions(tr_ev)
        prep_time = time.perf_counter() - t0
        # the modelling data = the filtered interactions of exactly these users
        inter = base_interactions[base_interactions["visitorid"].isin(sample_users)].reset_index(drop=True)
        if len(inter) == 0:
            print("   no interactions for this sample; skipped")
            continue
        gt, _ = build_ground_truth(te_ev, inter)

        t0 = time.perf_counter(); cf = ItemCFRecommender().fit(inter); cf_time = time.perf_counter() - t0

        gc.collect(); m0 = rss_mb()
        t0 = time.perf_counter(); Gs = build_bipartite_graph(inter); g_time = time.perf_counter() - t0
        g_mem = rss_mb() - m0

        pop = popularity_ranking(inter)
        methods = [cf, CommonNeighborsRecommender(Gs, pop), PersonalizedPageRankRecommender(Gs, pop)]
        users_gt = np.array(sorted(gt.keys()))
        eval_users = sorted(np.random.default_rng(SEED).choice(users_gt, size=min(n_eval, len(users_gt)),
                                                                 replace=False).tolist()) if len(users_gt) else []
        row = {"fraction": frac, "dataset_pct": int(frac * 100),
               "raw_events": len(ev), "users": inter["visitorid"].nunique(), "products": inter["itemid"].nunique(),
               "interactions": len(inter), "graph_nodes": Gs.number_of_nodes(), "graph_edges": Gs.number_of_edges(),
               "preprocess_time_s": prep_time, "cf_build_time_s": cf_time, "cf_similarity_mb": cf.similarity_matrix_mb(),
               "graph_build_time_s": g_time, "graph_memory_mb": g_mem, "process_rss_mb": rss_mb(),
               "n_eval_users": len(eval_users)}
        for m in methods:
            if eval_users:
                mdf, _ = evaluate_recommender(m, eval_users, gt, progress_every=0)
                row[f"{m.name}|time_ms"] = mdf["time_ms"].mean()
                row[f"{m.name}|precision@5"] = mdf["precision@5"].mean()
                row[f"{m.name}|ndcg@5"] = mdf["ndcg@5"].mean()
            else:
                row[f"{m.name}|time_ms"] = row[f"{m.name}|precision@5"] = row[f"{m.name}|ndcg@5"] = np.nan
        results.append(row)
        print(f"   users={row['users']:,} products={row['products']:,} edges={row['graph_edges']:,} | "
              f"eval users={len(eval_users)} | graph build {g_time:.2f}s, {g_mem:+.0f} MB | "
              f"PPR {row['Personalized PageRank|time_ms']:.1f} ms/user")
        del Gs, cf, methods, inter, ev, tr_ev, te_ev
        gc.collect()
    return pd.DataFrame(results)


t_start = time.perf_counter()
scalability_results = run_scalability(train_interactions, events)
print(f"\nScalability experiment finished in {time.perf_counter()-t_start:.0f}s")
display(scalability_results)

In [ ]:
# ---------------------------------------------------------------
# Scalability charts (the four requested + per-method breakdown)
# ---------------------------------------------------------------
sr = scalability_results
names = ["Item-Item CF", "Common Neighbors", "Personalized PageRank"]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (1) Dataset size vs recommendation time
for m in names:
    axes[0, 0].plot(sr["interactions"], sr[f"{m}|time_ms"], "o-", label=m, color=METHOD_COLORS[m])
axes[0, 0].set_title("Dataset size vs recommendation time"); axes[0, 0].set_xlabel("interactions (edges)")
axes[0, 0].set_ylabel("avg ms per user"); axes[0, 0].set_yscale("log"); axes[0, 0].legend()

# (2) Graph nodes vs memory
axes[0, 1].plot(sr["graph_nodes"], sr["graph_memory_mb"], "o-", color="#C44E52", label="NetworkX graph (RSS change)")
axes[0, 1].plot(sr["graph_nodes"], sr["cf_similarity_mb"], "s--", color="#4C72B0", label="CF sparse similarity matrix")
axes[0, 1].set_title("Graph nodes vs memory"); axes[0, 1].set_xlabel("graph nodes"); axes[0, 1].set_ylabel("MB")
axes[0, 1].legend()

# (3) Dataset size vs recommendation quality
for m in names:
    axes[1, 0].plot(sr["dataset_pct"], sr[f"{m}|ndcg@5"], "o-", label=m, color=METHOD_COLORS[m])
axes[1, 0].set_title("Dataset size vs recommendation quality"); axes[1, 0].set_xlabel("% of users")
axes[1, 0].set_ylabel("NDCG@5"); axes[1, 0].legend()

# (4) Graph construction time vs dataset size
axes[1, 1].plot(sr["interactions"], sr["graph_build_time_s"], "o-", color="#55A868", label="graph construction")
axes[1, 1].plot(sr["interactions"], sr["cf_build_time_s"], "s--", color="#4C72B0", label="CF similarity build")
axes[1, 1].set_title("Build time vs dataset size"); axes[1, 1].set_xlabel("interactions (edges)")
axes[1, 1].set_ylabel("seconds"); axes[1, 1].legend()

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("06_scalability.png")

# Interactive versions
time_long = sr.melt(id_vars=["dataset_pct", "interactions"], value_vars=[f"{m}|time_ms" for m in names],
                    var_name="method", value_name="ms_per_user")
time_long["method"] = time_long["method"].str.split("|").str[0]
save_plotly(px.line(time_long, x="interactions", y="ms_per_user", color="method", markers=True, log_y=True,
                    color_discrete_map=METHOD_COLORS, title="Dataset size vs recommendation time"),
            "06a_size_vs_time.html")
save_plotly(px.line(sr, x="graph_nodes", y="graph_memory_mb", markers=True, title="Graph nodes vs memory (MB)"),
            "06b_nodes_vs_memory.html")
q_long = sr.melt(id_vars=["dataset_pct"], value_vars=[f"{m}|ndcg@5" for m in names], var_name="method", value_name="ndcg@5")
q_long["method"] = q_long["method"].str.split("|").str[0]
save_plotly(px.line(q_long, x="dataset_pct", y="ndcg@5", color="method", markers=True,
                    color_discrete_map=METHOD_COLORS, title="Dataset size (% users) vs NDCG@5"),
            "06c_size_vs_quality.html")
save_plotly(px.line(sr, x="interactions", y="graph_build_time_s", markers=True, title="Graph construction time vs dataset size"),
            "06d_graph_build_time.html")

# Growth factors from the smallest to the largest run
if len(sr) >= 2:
    a, b = sr.iloc[0], sr.iloc[-1]
    print(f"From {a.dataset_pct}% to {b.dataset_pct}% of users: edges x{b.interactions/a.interactions:.1f}, "
          f"graph build time x{b.graph_build_time_s/max(a.graph_build_time_s,1e-9):.1f}, "
          f"PPR time/user x{b['Personalized PageRank|time_ms']/max(a['Personalized PageRank|time_ms'],1e-9):.1f}, "
          f"CN time/user x{b['Common Neighbors|time_ms']/max(a['Common Neighbors|time_ms'],1e-9):.1f}, "
          f"CF time/user x{b['Item-Item CF|time_ms']/max(a['Item-Item CF|time_ms'],1e-9):.1f}")

### Scalability limitations of NetworkX (an honest assessment)
- **In-memory, single machine.** NetworkX stores the whole graph as Python dictionaries in RAM (dict of dicts, one entry per edge in each direction). Every node and edge is a Python object, costing hundreds of bytes compared with a few bytes in a compact array. Memory therefore grows linearly with edges, but with a large constant. The "graph nodes vs memory" chart shows this.
- **Single-threaded and pure Python.** Graph construction and traversal (Common Neighbors) run in the Python interpreter on a single CPU core.
- **PPR cost per query is O(edges × iterations).** NetworkX's `pagerank` converts the graph to a sparse matrix and runs power iteration over **all** nodes for **every** user. Recommendation time grows with graph size even though one user only cares about their local neighbourhood. This is usually the steepest line in the time chart.
- **Common Neighbors cost depends on hubs.** One very popular product connects a user to thousands of others, so 2-hop expansion gets expensive as data grows.
- **What this project does *not* claim:** this is **not** a distributed system. Everything runs on one Colab CPU VM. Beyond what one machine's RAM can hold, you would need tools such as Spark GraphX/GraphFrames, a graph database (Neo4j), approximate PPR (push-based / Monte-Carlo random walks), or C/C++ graph libraries (igraph, graph-tool). Those are future work, not part of this implementation.

What we **did** do to stay within one machine: filtering, sparse matrices for CF, top-N neighbour pruning, chunked similarity computation, and evaluation on a reproducible user sample.

---
# 10. Backend Validation (Tests)
Automated checks that the pipeline behaves correctly. Each test prints PASS/FAIL, and the results are saved to `results/test_results.csv`. **Fix every failure before moving on to Phase 2.**

In [ ]:
# ---------------------------------------------------------------
# Test suite
# ---------------------------------------------------------------
TEST_RESULTS = []

def run_test(name, fn):
    try:
        fn()
        TEST_RESULTS.append({"test": name, "status": "PASS", "detail": ""})
        print(f"PASS  {name}")
    except AssertionError as e:
        TEST_RESULTS.append({"test": name, "status": "FAIL", "detail": str(e)})
        print(f"FAIL  {name}: {e}")
    except Exception as e:
        TEST_RESULTS.append({"test": name, "status": "ERROR", "detail": f"{type(e).__name__}: {e}"})
        print(f"ERROR {name}: {type(e).__name__}: {e}")

catalog = set(train_interactions["itemid"].unique())
train_seen = train_interactions.groupby("visitorid")["itemid"].agg(set).to_dict()
rng_t = np.random.default_rng(SEED)
TEST_USERS = rng_t.choice(np.array(sorted(train_seen)), size=min(30, len(train_seen)), replace=False).tolist()

# ---- Preprocessing ----
def t_clean():
    assert events[["timestamp", "visitorid", "itemid", "event"]].notna().all().all(), "nulls in key columns"
    assert set(events["weight"].unique()) <= set(EVENT_WEIGHTS.values()), "unexpected weight values"
    assert not events.duplicated(subset=REQUIRED_COLUMNS).any(), "duplicates remain"
    assert pd.api.types.is_datetime64_any_dtype(events["datetime"]), "datetime not converted"
    assert events["timestamp"].is_monotonic_increasing, "events not sorted by time"
def t_weights_map():
    for ev_name, w in EVENT_WEIGHTS.items():
        sub = events[events["event"] == ev_name]
        assert len(sub) == 0 or (sub["weight"] == w).all(), f"{ev_name} weight wrong"
def t_split():
    assert train_events["timestamp"].max() < CUTOFF_TS <= test_events["timestamp"].min(), "time leakage"
    assert len(train_events) + len(test_events) == len(events), "rows lost in split"
def t_filter():
    ud = train_interactions.groupby("visitorid")["itemid"].nunique()
    idg = train_interactions.groupby("itemid")["visitorid"].nunique()
    assert ud.min() >= MIN_USER_ITEMS and ud.max() <= MAX_USER_ITEMS, "user threshold violated"
    assert idg.min() >= MIN_ITEM_USERS, "item threshold violated"
    assert not train_interactions.duplicated(["visitorid", "itemid"]).any(), "pairs not aggregated"
    assert train_interactions["weight"].between(1, MAX_EDGE_WEIGHT).all(), "edge weight out of range"
def t_ground_truth():
    for u, rel in ground_truth.items():
        assert rel <= catalog, f"user {u} has relevant items outside the train catalogue"
        assert not (rel & train_seen[u]), f"user {u} relevant set overlaps train history"

# ---- Graph ----
def t_graph():
    assert nx.is_bipartite(G), "graph is not bipartite"
    assert graph_stats["user_nodes"] == train_interactions["visitorid"].nunique()
    assert graph_stats["product_nodes"] == train_interactions["itemid"].nunique()
    assert G.number_of_edges() == len(train_interactions), "edge count != interaction pairs"
    for a, b in G.edges():
        assert G.nodes[a]["node_type"] != G.nodes[b]["node_type"], "edge inside one partition"
def t_graph_weights():
    sample = train_interactions.sample(min(200, len(train_interactions)), random_state=SEED)
    for r in sample.itertuples():
        assert abs(G[user_node(r.visitorid)][product_node(r.itemid)]["weight"] - r.weight) < 1e-6

# ---- Recommenders ----
def make_rec_test(model):
    def _t():
        for u in TEST_USERS:
            recs = model.recommend(u, TOP_K)
            items = [i for i, _ in recs]
            scores = [s for _, s in recs]
            assert len(items) == TOP_K, f"{model.name}: {len(items)} recs for user {u}"
            assert len(set(items)) == len(items), f"{model.name}: duplicate recs"
            assert set(items) <= catalog, f"{model.name}: recommended item not in catalogue"
            assert not (set(items) & train_seen[u]), f"{model.name}: recommended an already-seen item to {u}"
            assert all(scores[i] >= scores[i + 1] - 1e-12 for i in range(len(scores) - 1)), f"{model.name}: scores not sorted"
    return _t
def t_cold_start():
    unknown = int(max(events["visitorid"].max(), 0) + 10**9)
    for m in [cf_model, cn_model, ppr_model]:
        recs = m.recommend(unknown, TOP_K)
        assert len(recs) == TOP_K and m.last_info["fallback"], f"{m.name}: cold-start fallback failed"
def t_toy():
    for m in toy_models:
        assert m.recommend(1, 1)[0][0] == 103, f"{m.name}: expected Charger for Asha in toy example"
def t_ppr_sums_to_one():
    pr = ppr_model.scores_for(user_node(TEST_USERS[0]))
    assert abs(sum(pr.values()) - 1) < 1e-4, "PPR scores do not sum to 1"

# ---- Metrics ----
def t_metrics():
    rec, rel = [10, 20, 30, 40, 50], {20, 99}
    assert precision_at_k(rec, rel) == 0.2 and recall_at_k(rec, rel) == 0.5
    assert abs(f1_at_k(rec, rel) - 2 * 0.2 * 0.5 / 0.7) < 1e-9
    assert abs(ndcg_at_k([1, 2, 3, 4, 5], {1, 2}) - 1.0) < 1e-9, "perfect ranking must give NDCG 1"
    assert ndcg_at_k([9, 8, 7, 6, 5], {1}) == 0.0
    assert ndcg_at_k([9, 1, 7, 6, 5], {1}) < ndcg_at_k([1, 9, 7, 6, 5], {1}), "NDCG must reward higher ranks"
def t_eval_results():
    for mtr in METRICS:
        assert per_user_eval[mtr].between(0, 1).all(), f"{mtr} out of [0,1]"
    assert (per_user_eval["time_ms"] > 0).all()
    assert per_user_eval.groupby("method")["visitorid"].apply(lambda s: sorted(s) == EVAL_USERS).all(), \
        "methods were not evaluated on identical users"
    hits = all_recommendations.groupby(["visitorid", "method"])["is_hit"].sum().reset_index()
    merged = hits.merge(per_user_eval, on=["visitorid", "method"])
    assert np.allclose(merged["is_hit"] / TOP_K, merged["precision@5"]), "precision inconsistent with hits"

# ---- Scalability ----
def t_scalability():
    s = scalability_results.sort_values("fraction")
    assert len(s) == len(SCALE_FRACTIONS), "some scalability runs were skipped"
    assert s["raw_events"].is_monotonic_increasing and s["users"].is_monotonic_increasing, "nested samples should grow"
    assert (s["graph_build_time_s"] > 0).all() and (s["graph_nodes"] > 0).all()
    full = s[s["fraction"] == 1.0].iloc[0]
    assert full["interactions"] == len(train_interactions), "100% run should reproduce the main pipeline"

for name, fn in [
    ("Preprocessing: clean events", t_clean), ("Preprocessing: event weights", t_weights_map),
    ("Split: chronological, no leakage", t_split), ("Filtering: thresholds & aggregation", t_filter),
    ("Ground truth: new, in-catalogue items only", t_ground_truth),
    ("Graph: bipartite structure & counts", t_graph), ("Graph: edge weights match data", t_graph_weights),
    ("Item-Item CF: valid recommendations", make_rec_test(cf_model)),
    ("Common Neighbors: valid recommendations", make_rec_test(cn_model)),
    ("Personalized PageRank: valid recommendations", make_rec_test(ppr_model)),
    ("Popularity: valid recommendations", make_rec_test(pop_model)),
    ("Cold-start fallback", t_cold_start), ("Toy example correctness", t_toy),
    ("PPR is a probability distribution", t_ppr_sums_to_one),
    ("Metrics: hand-computed values", t_metrics), ("Evaluation: consistency", t_eval_results),
    ("Scalability: measurements", t_scalability),
]:
    run_test(name, fn)

test_results = pd.DataFrame(TEST_RESULTS)
n_pass = (test_results["status"] == "PASS").sum()
print(f"\n{n_pass}/{len(test_results)} tests passed")

---
# 11. Export Outputs for Phase 2
Everything the Streamlit dashboard needs is saved under `outputs/`, using the same folder names as the Phase 2 project:

```
outputs/
├── data/            train_interactions.csv.gz, test_ground_truth.csv, item_popularity.csv, eval_user_histories.csv
├── results/         evaluation_summary.csv, per_user_evaluation.csv, recommendations.csv, scalability_results.csv,
│                    graph_stats.json, dataset_overview.json, test_results.csv, interpretation.md, config.json
├── models/          bipartite_graph.pkl, item_similarity.npz, cf_index.json
└── visualizations/  all PNG charts + interactive Plotly HTML files
```

In [ ]:
# ---------------------------------------------------------------
# Save all reusable outputs
# ---------------------------------------------------------------
def to_jsonable(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, (np.integer,)): v = int(v)
        elif isinstance(v, (np.floating,)): v = float(v)
        elif isinstance(v, (np.bool_,)): v = bool(v)
        out[k] = v
    return out

# --- data/ ---
train_interactions.to_csv(os.path.join(DATA_OUT, "train_interactions.csv.gz"), index=False, compression="gzip")
pd.DataFrame([(u, i) for u, items in ground_truth.items() for i in sorted(items)],
             columns=["visitorid", "itemid"]).to_csv(os.path.join(DATA_OUT, "test_ground_truth.csv"), index=False)
(train_interactions.groupby("itemid").agg(n_users=("visitorid", "nunique"), total_weight=("weight", "sum"))
 .reset_index().sort_values("n_users", ascending=False)
 .to_csv(os.path.join(DATA_OUT, "item_popularity.csv"), index=False))
train_interactions[train_interactions["visitorid"].isin(EVAL_USERS)].to_csv(
    os.path.join(DATA_OUT, "eval_user_histories.csv"), index=False)

# --- results/ ---
evaluation_summary.to_csv(os.path.join(RESULTS_DIR, "evaluation_summary.csv"), index=False)
per_user_eval.to_csv(os.path.join(RESULTS_DIR, "per_user_evaluation.csv"), index=False)
all_recommendations.to_csv(os.path.join(RESULTS_DIR, "recommendations.csv"), index=False)
scalability_results.to_csv(os.path.join(RESULTS_DIR, "scalability_results.csv"), index=False)
test_results.to_csv(os.path.join(RESULTS_DIR, "test_results.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "graph_stats.json"), "w") as f:
    json.dump(to_jsonable(graph_stats), f, indent=2)
with open(os.path.join(RESULTS_DIR, "dataset_overview.json"), "w") as f:
    json.dump({"raw": to_jsonable(dataset_overview), "cleaning": to_jsonable(cleaning_log),
               "event_distribution": {k: int(v) for k, v in event_counts.items()},
               "clean_train_summary": to_jsonable(clean_summary),
               "cutoff_datetime": str(pd.to_datetime(CUTOFF_TS, unit="ms")), "demo_user": DEMO_USER}, f, indent=2)
with open(os.path.join(RESULTS_DIR, "interpretation.md"), "w") as f:
    f.write(INTERPRETATION)
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, dict, bool))
          and k not in {"INTERPRETATION"}}
with open(os.path.join(RESULTS_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2, default=str)

# --- models/ ---
with open(os.path.join(MODELS_DIR, "bipartite_graph.pkl"), "wb") as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)
sp.save_npz(os.path.join(MODELS_DIR, "item_similarity.npz"), cf_model.S)
with open(os.path.join(MODELS_DIR, "cf_index.json"), "w") as f:
    json.dump({"user_ids": cf_model.user_ids.tolist(), "item_ids": cf_model.item_ids.tolist()}, f)

# --- listing ---
total = 0
for root, _, files_ in os.walk(OUTPUT_DIR):
    for fn in sorted(files_):
        p = os.path.join(root, fn); size = os.path.getsize(p); total += size
        print(f"{os.path.relpath(p, OUTPUT_DIR):<50} {size/1e6:8.2f} MB")
print(f"{'TOTAL':<50} {total/1e6:8.2f} MB")

In [ ]:
# ---------------------------------------------------------------
# Zip and download (Colab), optionally copy to Google Drive
# ---------------------------------------------------------------
COPY_TO_DRIVE = False   # set True to also keep a copy in Drive (survives the Colab session)
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/retailrocket_outputs"

zip_base = os.path.join(os.path.dirname(os.path.abspath(OUTPUT_DIR)), "outputs")
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print(f"Created {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

if IN_COLAB:
    if COPY_TO_DRIVE:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
        shutil.copy(zip_path, DRIVE_EXPORT_DIR)
        print(f"Copied to {DRIVE_EXPORT_DIR}")
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print(f"Automatic download failed ({e}). Use the Files panel on the left: right-click outputs.zip -> Download.")

### How to reuse these outputs in Phase 2
1. Unzip `outputs.zip` on your PC.
2. Copy the folders into the VS Code project: `data/` → `recommendation-engine/data/`, `results/` → `recommendation-engine/results/`, `visualizations/` → `recommendation-engine/visualizations/`, `models/` → `recommendation-engine/data/models/` (or keep a `models/` folder).
3. The classes in this notebook map directly onto the Phase 2 modules:

| Notebook section | Phase 2 file |
|---|---|
| `aggregate_interactions`, `filter_interactions`, `build_ground_truth` | `src/data_preprocessing.py` |
| `ItemCFRecommender`, `PopularityRecommender`, popularity helpers | `src/baseline_recommender.py` |
| `build_bipartite_graph`, `graph_statistics`, `extract_subgraph` | `src/graph_builder.py` |
| `CommonNeighborsRecommender`, `PersonalizedPageRankRecommender` | `src/graph_recommender.py` |
| metric functions, `evaluate_recommender`, `bootstrap_ci` | `src/evaluation.py` |
| `run_scalability` | `src/scalability.py` |

4. The dashboard can **load** the saved metrics and charts, and **compute live** recommendations from `bipartite_graph.pkl` + `item_similarity.npz`, so nothing is faked.

Make sure Section 10 shows all tests passing before you start Phase 2.

---
# 12. Viva Notes (simple explanations)

**Q: Why a graph?** User-product interactions naturally form a network. A graph lets us use *indirect* connections (a user → a product → other users → their products), not only direct overlaps.

**Q: What is a bipartite graph?** A graph with two node types where edges only connect the two types. Here they are users and products, and a user never links directly to another user.

**Q: What is implicit feedback?** Behaviour (views, carts, purchases) instead of explicit ratings. It is abundant but noisy, which is why we weight it.

**Q: Why cosine similarity?** It compares the *pattern* of who interacted with two products and ignores how popular each product is overall, so very popular items do not automatically look similar to everything.

**Q: Common Neighbors vs PageRank?** CN counts shared products (local, 2-hop, fast). PPR simulates a random walk that keeps restarting at the user, so it captures multi-hop structure and is more global, but costs more per query.

**Q: What does alpha = 0.85 mean?** At each step the walker follows an edge with 85% probability and jumps back to the target user with 15% probability. A lower alpha keeps results closer to the user's immediate neighbourhood.

**Q: Why a chronological split?** To simulate real use (predict the future from the past) and avoid data leakage, which would inflate the scores.

**Q: Why are the metric values so low?** There are tens of thousands of products, and most users touch only a handful. Getting even one of 5 guesses right is hard. We judge methods by comparing them on the same split, and we check significance with confidence intervals.

**Q: Precision vs Recall vs NDCG?** Precision: how many of my 5 picks were right. Recall: how much of what the user did later I captured. NDCG: like precision, but correct items ranked higher earn more credit.

**Q: Is this system distributed?** No. It runs on one machine using NetworkX, which works in memory. We measured how time and memory grow with data size and explained the limits. Distributed graph processing (e.g. Spark GraphFrames) would be the next step.

**Q: What is the cold-start problem?** New users or products have no history. Our recommenders fall back to popular items for users without enough history. New products cannot be recommended until they appear in the training data.